# CA1: Variational Autoencoders

Clean, reproducible notebook following course rules: overview → setup/config → data → model → training → evaluation → analysis.


## Overview & Objectives
- Implement convolutional beta-VAE for 128×128 RGB faces.
- Keep configuration centralized and deterministic (seeds + CuDNN flags).
- Provide smoke-run defaults, with hooks for full 1000-epoch training.
- Keep outputs light; visual artifacts live under `images/`.
- Sync with report/README after code or experiment changes.


## Notebook Roadmap
1. Setup & reproducibility (imports, seeds, config, device).
2. Data loading (transforms, deterministic train/val split).
3. Model (encoder/decoder, reparameterization).
4. Loss (ELBO with KL + reconstruction, beta scaling).
5. Training utilities (epoch loops, logging, sample saving).
6. Optional run cells (smoke first, then full).
7. Evaluation helpers (recon/gen grids).
8. Notes on syncing figures and report.


## Setup and Reproducibility


In [ ]:
from __future__ import annotations

import math
import random
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.utils import save_image

# Paths
ASSIGNMENT_ROOT = Path('..').resolve()
DATA_ROOT = ASSIGNMENT_ROOT / 'train'
OUT_DIR = ASSIGNMENT_ROOT / 'output'

# Global config (adjust epochs for full training; defaults are smoke-friendly)
config: Dict[str, object] = {
    'image_size': 128,
    'batch_size': 32,
    'val_split': 0.2,
    'epochs': 5,  # set to 1000 for full run
    'lr': 5e-4,
    'beta': 1.0,
    'num_workers': 4,
    'sample_grid': 16,
    'seed': 42,
}

def set_seed(seed: int) -> None:
    """Set random seeds for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(config['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
assert DATA_ROOT.exists(), f'Dataset not found at {DATA_ROOT}'
OUT_DIR.mkdir(parents=True, exist_ok=True)



## Data Loading


In [ ]:
def get_dataloaders(cfg: Dict[str, object]) -> Tuple[DataLoader, DataLoader]:
    """Create train and validation DataLoaders with deterministic split and transforms."""
    tfm = transforms.Compose([
        transforms.Resize((cfg['image_size'], cfg['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    dataset = datasets.ImageFolder(root=str(DATA_ROOT), transform=tfm)
    val_len = int(len(dataset) * cfg['val_split'])
    train_len = len(dataset) - val_len
    generator = torch.Generator().manual_seed(cfg['seed'])
    train_set, val_set = random_split(dataset, [train_len, val_len], generator=generator)

    train_loader = DataLoader(
        train_set,
        batch_size=cfg['batch_size'],
        shuffle=True,
        num_workers=cfg['num_workers'],
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_set,
        batch_size=cfg['batch_size'],
        shuffle=False,
        num_workers=cfg['num_workers'],
        pin_memory=True,
    )
    return train_loader, val_loader

train_loader, val_loader = get_dataloaders(config)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')



## Model Definition (Beta-VAE)


In [ ]:
class BetaVAE(nn.Module):
    """Convolutional beta-VAE for 128x128 RGB images with configurable latent dimension and dropout."""
    def __init__(self, latent_dim: int = 32, dropout: float = 0.2):
        super().__init__()
        # Encoder: 128 -> 8 (4 downsamples)
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        hidden_dim = 256 * 8 * 8
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        self.fc_dec = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),
            nn.Tanh(),
        )

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encode input images to latent mean and log-variance."""
        h = self.enc(x)
        h = torch.flatten(h, start_dim=1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """Sample latent vector z using the reparameterization trick."""
        logvar = torch.clamp(logvar, min=-10, max=10)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode latent vector z to reconstructed image."""
        h = self.fc_dec(z)
        h = h.view(h.size(0), 256, 8, 8)
        return self.dec(h)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Forward pass: returns reconstruction, mean, and log-variance."""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar



## Loss Function (ELBO)


In [ ]:
def elbo_loss(
    recon: torch.Tensor,
    x: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    beta: float,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute ELBO loss: reconstruction + beta-scaled KL divergence."""
    recon_loss = F.mse_loss(recon, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    loss = recon_loss + beta * kl
    return loss, recon_loss, kl



## Training Utilities


In [ ]:
def train_one_epoch(model: BetaVAE, loader: DataLoader, optimizer, beta: float, device: torch.device):
    """Train model for one epoch and return running loss metrics."""
    model.train()
    running = {'loss': 0.0, 'recon': 0.0, 'kl': 0.0, 'count': 0}
    for imgs, _ in loader:
        imgs = imgs.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = model(imgs)
        loss, r_loss, kl = elbo_loss(recon, imgs, mu, logvar, beta)
        loss.backward()
        optimizer.step()
        bs = imgs.size(0)
        running['loss'] += loss.item() * bs
        running['recon'] += r_loss.item() * bs
        running['kl'] += kl.item() * bs
        running['count'] += bs
    for k in ['loss', 'recon', 'kl']:
        running[k] /= max(1, running['count'])
    return running

def eval_epoch(model: BetaVAE, loader: DataLoader, beta: float, device: torch.device):
    """Evaluate model for one epoch and return running loss metrics."""
    model.eval()
    running = {'loss': 0.0, 'recon': 0.0, 'kl': 0.0, 'count': 0}
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            recon, mu, logvar = model(imgs)
            loss, r_loss, kl = elbo_loss(recon, imgs, mu, logvar, beta)
            bs = imgs.size(0)
            running['loss'] += loss.item() * bs
            running['recon'] += r_loss.item() * bs
            running['kl'] += kl.item() * bs
            running['count'] += bs
    for k in ['loss', 'recon', 'kl']:
        running[k] /= max(1, running['count'])
    return running

def save_samples(model: BetaVAE, device: torch.device, data: torch.Tensor, out_dir: Path, step_label: str, num_samples: int) -> None:
    """Save reconstruction and generation image grids for a batch of data and random samples."""
    out_dir.mkdir(parents=True, exist_ok=True)
    model.eval()
    with torch.no_grad():
        data = data.to(device)[:num_samples]
        recon, _, _ = model(data)
        random_z = torch.randn(num_samples, model.fc_mu.out_features, device=device)
        gen = model.decode(random_z)
        save_image(
            torch.cat([data, recon], dim=0) * 0.5 + 0.5,
            out_dir / f'recon_{step_label}.png',
            nrow=int(math.sqrt(num_samples * 2)),
        )
        save_image(
            gen * 0.5 + 0.5,
            out_dir / f'gen_{step_label}.png',
            nrow=int(math.sqrt(num_samples)),
        )
    model.train()

def run_training(cfg: Dict[str, object]):
    """Run full training loop for beta-VAE and save periodic samples."""
    model = BetaVAE().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    history = []
    for epoch in range(1, int(cfg['epochs']) + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, beta=cfg['beta'], device=device)
        val_metrics = eval_epoch(model, val_loader, beta=cfg['beta'], device=device)
        history.append({'epoch': epoch, 'train': train_metrics, 'val': val_metrics})
        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | train loss {train_metrics['loss']:.4f} "
                f"(recon {train_metrics['recon']:.4f}, kl {train_metrics['kl']:.4f}) | "
                f"val loss {val_metrics['loss']:.4f}"
            )
            sample_batch, _ = next(iter(train_loader))
            save_samples(
                model,
                device,
                sample_batch,
                OUT_DIR,
                step_label=f'epoch{epoch}',
                num_samples=min(cfg['sample_grid'], sample_batch.size(0)),
            )
    return model, history



## Run (smoke first)
- Defaults use 5 epochs for a quick smoke test (loss finite, shapes valid).
- For the full assignment run, set `config['epochs'] = 1000` and consider lowering `num_workers` on CPU-only runs.
- Outputs land in `output/` (recon/gen grids).



In [ ]:
# Canonical execution is script-first.
# This cell runs the full CA1 pipeline from the project root using the fixed venv.
!bash -lc "source /Users/tahamajs/Documents/uni/venv/bin/activate && export MPLCONFIGDIR=/tmp/matplotlib && mkdir -p /tmp/matplotlib && cd /Users/tahamajs/Documents/uni/DGM/CA1_Variational_Autoencoders && python code/vae_training.py --data-root train --out-dir output --images-dir images --checkpoint-dir output/checkpoints --epochs 120 --batch-size 64 --lr 5e-4 --beta 1.0 --device auto --num-workers 0 --analysis-samples 500 --save-report-plots"

# For a quick validation pass, replace --epochs 120 with --epochs 1.


## Evaluation Helpers
Use these after training to visualize reconstructions/generations without rerunning heavy cells.


In [ ]:
def preview_reconstructions(model: BetaVAE, loader: DataLoader, max_images: int = 16):
    """Save preview grid of reconstructions for a batch from the loader."""
    model.eval()
    with torch.no_grad():
        imgs, _ = next(iter(loader))
        imgs = imgs.to(device)
        recon, _, _ = model(imgs[:max_images])
        save_samples(model, device, imgs, OUT_DIR, step_label='preview', num_samples=min(max_images, imgs.size(0)))

# Example usage after training:
# preview_reconstructions(model, val_loader)



## Analysis and Report Sync
- Use generated `output/recon_*.png` and `output/gen_*.png` for figures (move to `images/` if needed).
- Keep README/report metrics aligned with the latest run (note epochs, beta, batch size, seed, commit hash).
- For latent visualizations (t-SNE/PCA), reuse the encoded features via `model.encode` on batches; add as needed without duplicating training logic.


## Checklist
- [ ] Seeds set (`set_seed`), deterministic flags enabled when on CUDA.
- [ ] Config centralized; no hardcoded hyperparameters elsewhere.
- [ ] Smoke test run (5 epochs) completes with finite loss.
- [ ] No heavy outputs embedded; artifacts saved to disk.
- [ ] Report/README updated if experiments change.
- [ ] Commit changes with message describing notebook restructuring.
